# Frenet scale ablation — 5,000 training scenarios × 50 epochs

Trains the best-ever Frenet configuration at the full 5,000-scenario scale, seed=269, used to validate that the Frenet penalty does not close at increased data scale.

**Best-ever Frenet configuration**
- `model.kinematic=frenet`
- `normalization_stats=frenet_norm_stats_v1` (paired to `tanh(d/3)` output range)
- `FRENET_SMART_CENTERLINE=1`, `FRENET_TANH_D=1`, `FRENET_TANH_D_SCALE=3.0`
- CenterlineEncoder ON at training and evaluation

**Expected wall-clock**: 1.5–2h on L4, 3–4h on T4.

**Setup requirements**: see `QUICKSTART.md` at the repo root.

## 1. Environment detection + paths

In [ ]:
import os, sys, shutil, subprocess, json, pathlib, time

# ---------- environment detection ----------
IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IS_COLAB_ENTERPRISE = os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE'
IS_BARE_VM = not IS_COLAB and not IS_COLAB_ENTERPRISE

# ---------- the key knobs ----------
SCALE_POINTS         = [5000]          # train-set sizes
EPOCH_POINTS         = [50]            # epochs per training
KINEMATICS_TO_RUN    = ['frenet']      # Frenet only — teammates run controls
SCALE_SEED           = 269
HELDOUT_SCENARIOS_SCALE = 5000         # large eval set
MAX_SCENARIOS_TO_PREPROCESS = max(SCALE_POINTS)
TRAIN_BATCH_SIZE     = 32
TRAIN_EPOCHS         = max(EPOCH_POINTS)
WARM_UP_EPOCHS       = min(5, max(0, TRAIN_EPOCHS - 1))

# ---------- python interpreter ----------
PYTHON = '/usr/bin/python3'   # use system python (has all deps)
PIP    = '/usr/bin/pip3'

# ---------- path setup ----------
if IS_BARE_VM:
    print('Bare GCP VM detected')
    DRIVE_FOLDER = '/content/drive/MyDrive/cs269'
    pathlib.Path('/content').mkdir(parents=True, exist_ok=True)
    pathlib.Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
    REPO_DIR = '/content/CS269FlowPlannerProject'
    if not pathlib.Path(REPO_DIR).exists():
        # symlink to wherever the user cloned it
        for cand in [os.path.expanduser('~/CS269FlowPlannerProject'),
                     os.path.expanduser('~/269PROJ'),
                     str(pathlib.Path.cwd())]:
            if pathlib.Path(cand).exists() and (pathlib.Path(cand) / 'notebooks').exists():
                subprocess.run(['ln', '-sfn', cand, REPO_DIR], check=False)
                print(f'  symlinked {REPO_DIR} -> {cand}')
                break
elif IS_COLAB_ENTERPRISE:
    print('Colab Enterprise detected')
    DRIVE_FOLDER = '/content/drive/MyDrive/cs269'
    pathlib.Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
    REPO_DIR = '/content/CS269FlowPlannerProject'
elif IS_COLAB:
    print('Standard Colab detected')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'  drive.mount failed: {e}')
    DRIVE_FOLDER = '/content/drive/MyDrive/cs269'
    REPO_DIR = '/content/CS269FlowPlannerProject'
    if not pathlib.Path(REPO_DIR).exists():
        subprocess.run(['git', 'clone', 'https://github.com/wimaan3/CS269FlowPlannerProject.git', REPO_DIR], check=True)

# ---------- per-run paths ----------
FP_DIR              = f'{REPO_DIR}/flow_planner'
LOCAL_WORK          = '/content/work'
LOCAL_NUPLAN        = f'{LOCAL_WORK}/nuplan'
LOCAL_MAPS          = f'{LOCAL_NUPLAN}/maps'
LOCAL_LOGS          = f'{LOCAL_NUPLAN}/data/cache/mini'
LOCAL_EXP           = f'{LOCAL_NUPLAN}/exp'
LOCAL_CACHE         = f'{LOCAL_WORK}/preprocessed_cache'
LOCAL_HELDOUT_CACHE = f'{LOCAL_WORK}/preprocessed_cache_heldout'
LOCAL_RUNS          = f'{LOCAL_WORK}/runs'
LOCAL_TB            = f'{LOCAL_WORK}/tb'
LOG_SPLIT_JSON      = f'{LOCAL_WORK}/log_split.json'
DRIVE_ZIPS          = f'{DRIVE_FOLDER}/nuplan_zips'
DRIVE_OUT_SCALE     = f'{DRIVE_FOLDER}/scale_ablation_full'

for d in [LOCAL_WORK, LOCAL_NUPLAN, LOCAL_MAPS, LOCAL_LOGS, LOCAL_EXP,
          LOCAL_CACHE, LOCAL_HELDOUT_CACHE, LOCAL_RUNS, LOCAL_TB,
          DRIVE_ZIPS, DRIVE_OUT_SCALE,
          f'{DRIVE_OUT_SCALE}/checkpoints',
          f'{DRIVE_OUT_SCALE}/results',
          f'{DRIVE_OUT_SCALE}/logs',
          f'{DRIVE_OUT_SCALE}/manifests']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# ---------- env for nuPlan SDK ----------
os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP

print('\n--- config ---')
print(f'SCALE_POINTS={SCALE_POINTS}, EPOCH_POINTS={EPOCH_POINTS}, KIN={KINEMATICS_TO_RUN}')
print(f'PYTHON={PYTHON}')
print(f'REPO_DIR={REPO_DIR}')
print(f'FP_DIR={FP_DIR}')
print(f'DRIVE_FOLDER={DRIVE_FOLDER}')
print(f'LOCAL_CACHE={LOCAL_CACHE}')
print(f'LOCAL_HELDOUT_CACHE={LOCAL_HELDOUT_CACHE}')
print(f'DRIVE_OUT_SCALE={DRIVE_OUT_SCALE}')
print(f'Total trainings: {len(SCALE_POINTS) * len(KINEMATICS_TO_RUN) * len(EPOCH_POINTS)}')


## 2. Verify dependencies

In [ ]:
# Sanity-check that all critical deps are importable in /usr/bin/python3
required = ['torch', 'hydra', 'omegaconf', 'numpy', 'matplotlib']
for mod in required:
    r = subprocess.run([PYTHON, '-c', f'import {mod}; print({mod}.__version__)'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  {mod:<12} {r.stdout.strip()}')
    else:
        print(f'  [FAIL] {mod}: not installed — installing now')
        subprocess.run([PIP, 'install', '--quiet', mod], check=False)

# flow_planner package importable?
r = subprocess.run([PYTHON, '-c',
                    f'import sys; sys.path.insert(0, "{FP_DIR}"); '
                    'from flow_planner.data.dataset.nuplan import NuPlanDataset; print("OK")'],
                   capture_output=True, text=True)
print(f'  flow_planner: {r.stdout.strip() if r.returncode == 0 else r.stderr[-200:]}')


## 3. Download nuPlan zips (S3 or Drive)

In [ ]:
# Download nuplan-v1.1_mini.zip + nuplan-maps-v1.0.zip from S3 if not on Drive
S3_BASE = 's3://motional-nuplan/public/nuplan-v1.1'
ZIPS_NEEDED = [
    ('nuplan-v1.1_mini.zip', f'{S3_BASE}/nuplan-v1.1_mini.zip', 8_550_100_030, LOCAL_LOGS),
    ('nuplan-maps-v1.0.zip', f'{S3_BASE}/nuplan-maps-v1.0.zip', 971_557_640, LOCAL_MAPS),
]

# Check if aws CLI is available
if subprocess.run(['which', 'aws'], capture_output=True).returncode != 0:
    print('[WARN]  aws CLI not found — installing AWS CLI v2')
    subprocess.run(['sudo', 'apt-get', 'install', '-y', 'unzip'], check=False)
    subprocess.run(['bash', '-c',
        'cd /tmp && curl -s https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip -o awscliv2.zip && '
        'unzip -q -o awscliv2.zip && sudo ./aws/install --update'], check=False)

for name, s3_url, expected_size, _ in ZIPS_NEEDED:
    dst = f'{DRIVE_ZIPS}/{name}'
    if pathlib.Path(dst).exists() and pathlib.Path(dst).stat().st_size >= expected_size * 0.99:
        print(f'  [OK] {name} already present ({pathlib.Path(dst).stat().st_size/1e9:.1f} GB)')
        continue
    print(f'  ⬇️  downloading {name} from S3...')
    r = subprocess.run(['aws', 's3', 'cp', '--no-sign-request', s3_url, dst],
                       capture_output=False)
    if r.returncode != 0:
        raise RuntimeError(f'S3 download failed for {name}')
    print(f'  [OK] {name} downloaded')


## 4. Extract + flatten zip directory structure

In [ ]:
import zipfile

ZIP_TARGETS = [
    ('nuplan-v1.1_mini.zip', LOCAL_LOGS,  '*.db'),
    ('nuplan-maps-v1.0.zip', LOCAL_MAPS,  'nuplan-maps-v1.0.json'),
]

for name, target_dir, marker_pattern in ZIP_TARGETS:
    src = f'{DRIVE_ZIPS}/{name}'
    # Check if target already populated
    if marker_pattern.startswith('*'):
        already = list(pathlib.Path(target_dir).glob(marker_pattern))
    else:
        already = list(pathlib.Path(target_dir).glob(f'**/{marker_pattern}'))
    if already:
        print(f'  [OK] {name} already extracted ({len(already)} matching files in {target_dir})')
        continue
    print(f'  [PKG] extracting {name} -> {target_dir}')
    pathlib.Path(target_dir).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src) as zf:
        zf.extractall(target_dir)
    print(f'  [OK] {name} extracted')

# ---------- auto-flatten nested dirs ----------
def flatten_if_nested(parent, marker):
    p = pathlib.Path(parent)
    if not p.exists() or (p / marker).exists():
        return
    for sub in p.iterdir():
        if sub.is_dir() and (sub / marker).exists():
            print(f'  flattening {sub} -> {p}')
            for item in sub.iterdir():
                tgt = p / item.name
                if not tgt.exists():
                    shutil.move(str(item), str(tgt))
            try: sub.rmdir()
            except OSError: pass
            return

flatten_if_nested(LOCAL_MAPS, 'nuplan-maps-v1.0.json')

# .db files may be in nested subdirs
if not list(pathlib.Path(LOCAL_LOGS).glob('*.db')):
    for db in pathlib.Path(LOCAL_LOGS).rglob('*.db'):
        tgt = pathlib.Path(LOCAL_LOGS) / db.name
        if not tgt.exists():
            shutil.move(str(db), str(tgt))

# Verify
maps_ok = pathlib.Path(f'{LOCAL_MAPS}/nuplan-maps-v1.0.json').exists()
db_count = len(list(pathlib.Path(LOCAL_LOGS).glob('*.db')))
print(f'\n--- post-extract sanity ---')
print(f'  maps json: {maps_ok}')
print(f'  .db files: {db_count}')
assert maps_ok and db_count >= 50, f'Extract incomplete: maps_ok={maps_ok} db_count={db_count}'


## 5. Generate log split (seed 42, 54/10)

In [ ]:
if pathlib.Path(LOG_SPLIT_JSON).exists():
    print(f'  [OK] log split already exists at {LOG_SPLIT_JSON}')
else:
    print(f'  generating log split (54 train / 10 val from 64 logs, seed 42)')
    r = subprocess.run([PYTHON, f'{REPO_DIR}/scripts/generate_log_split.py',
                        '--logs_dir', LOCAL_LOGS,
                        '--output', LOG_SPLIT_JSON],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:])
        raise RuntimeError('log split generation failed')
    print(r.stdout)


## 6. Phase 1 — Preprocess 5000 training scenarios

In [ ]:
SCALE_CACHE_DRIVE = f'{DRIVE_FOLDER}/preprocessed_cache_{MAX_SCENARIOS_TO_PREPROCESS}'

scale_cache_path = pathlib.Path(SCALE_CACHE_DRIVE)
existing_drive_npz = sorted(scale_cache_path.glob('*.npz')) if scale_cache_path.exists() else []

if scale_cache_path.exists() and len(existing_drive_npz) >= MAX_SCENARIOS_TO_PREPROCESS * 0.95:
    print(f'  [OK] Drive cache has {len(existing_drive_npz)} .npz files — copying to local')
    pathlib.Path(LOCAL_CACHE).mkdir(parents=True, exist_ok=True)
    local_npz = sorted(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
    if len(local_npz) < len(existing_drive_npz) * 0.95:
        subprocess.run(['rsync', '-a', '--info=stats2',
                       f'{scale_cache_path}/', f'{LOCAL_CACHE}/'], check=True)
        print(f'  rsync done')
    else:
        print(f'  [OK] local cache already has {len(local_npz)} .npz files')
else:
    print(f'  preprocessing fresh to {MAX_SCENARIOS_TO_PREPROCESS} scenarios')
    pathlib.Path(LOCAL_CACHE).mkdir(parents=True, exist_ok=True)
    cmd = [
        PYTHON, '-m', 'flow_planner.run_script.preprocess',
        '--data_path',        LOCAL_LOGS,
        '--map_path',         LOCAL_MAPS,
        '--save_path',        LOCAL_CACHE,
        '--total_scenarios',  str(MAX_SCENARIOS_TO_PREPROCESS),
        '--log_names_json',   LOG_SPLIT_JSON,
        '--log_names_key',    'train',
        '--seed',             str(SCALE_SEED),
    ]
    # Run from FP_DIR so module imports work
    env = os.environ.copy()
    env['PYTHONPATH'] = FP_DIR + ':' + env.get('PYTHONPATH', '')
    r = subprocess.run(cmd, cwd=FP_DIR, env=env, capture_output=False)
    if r.returncode != 0:
        raise RuntimeError(f'preprocess failed (exit {r.returncode})')

    # Mirror to Drive for resilience
    scale_cache_path.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f'{LOCAL_CACHE}/', f'{SCALE_CACHE_DRIVE}/'], check=False)
    print(f'  [OK] mirrored to {SCALE_CACHE_DRIVE}')

# Build per-scale manifests
all_npz = sorted(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
print(f'\n  Final cache size: {len(all_npz)} .npz files')

for scale in SCALE_POINTS:
    manifest_name = f'diffusion_planner_training_{scale}.json'
    local_manifest = pathlib.Path(LOCAL_CACHE) / manifest_name
    drive_manifest = pathlib.Path(f'{DRIVE_OUT_SCALE}/manifests/train_{scale}.json')
    subset = [f.name for f in all_npz[:scale]]
    local_manifest.write_text(json.dumps(subset))
    drive_manifest.write_text(json.dumps(subset))
    print(f'  manifest for scale={scale}: {len(subset)} scenarios')

# Default manifest for downstream code
pathlib.Path(LOCAL_CACHE + '/diffusion_planner_training.json').write_text(
    json.dumps([f.name for f in all_npz[:max(SCALE_POINTS)]])
)
print('\n  [OK] Phase 1 complete')


## 7. Phase 1b — Preprocess 5000 heldout scenarios from val logs

In [ ]:
drive_heldout_dir = pathlib.Path(f'{DRIVE_FOLDER}/heldout_cache_{HELDOUT_SCENARIOS_SCALE}')

if drive_heldout_dir.exists() and len(list(drive_heldout_dir.glob('*.npz'))) >= HELDOUT_SCENARIOS_SCALE * 0.95:
    print(f'  [OK] Drive heldout cache exists — copying down')
    pathlib.Path(LOCAL_HELDOUT_CACHE).mkdir(parents=True, exist_ok=True)
    if not list(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz')):
        subprocess.run(['rsync', '-a', f'{drive_heldout_dir}/', f'{LOCAL_HELDOUT_CACHE}/'], check=True)
elif pathlib.Path(LOCAL_HELDOUT_CACHE).exists() and \
     len(list(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))) >= HELDOUT_SCENARIOS_SCALE * 0.95:
    print(f'  [OK] heldout cache already local')
else:
    print(f'  building heldout cache from val logs ({HELDOUT_SCENARIOS_SCALE} scenarios)')
    cmd = [
        PYTHON, '-m', 'flow_planner.run_script.preprocess',
        '--data_path',        LOCAL_LOGS,
        '--map_path',         LOCAL_MAPS,
        '--save_path',        LOCAL_HELDOUT_CACHE,
        '--total_scenarios',  str(HELDOUT_SCENARIOS_SCALE),
        '--log_names_json',   LOG_SPLIT_JSON,
        '--log_names_key',    'val',
        '--seed',             '0',
    ]
    env = os.environ.copy()
    env['PYTHONPATH'] = FP_DIR + ':' + env.get('PYTHONPATH', '')
    r = subprocess.run(cmd, cwd=FP_DIR, env=env, capture_output=False)
    if r.returncode != 0:
        raise RuntimeError(f'heldout preprocess failed (exit {r.returncode})')

    heldout_npz = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
    pathlib.Path(LOCAL_HELDOUT_CACHE + '/diffusion_planner_training.json').write_text(
        json.dumps([f.name for f in heldout_npz])
    )

    # Mirror to Drive
    drive_heldout_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f'{LOCAL_HELDOUT_CACHE}/', f'{drive_heldout_dir}/'], check=False)
    print(f'  [OK] mirrored to {drive_heldout_dir}')

heldout_npz_count = len(list(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz')))
print(f'\n  Heldout cache: {heldout_npz_count} .npz files')

# Make sure the manifest exists
heldout_manifest = pathlib.Path(LOCAL_HELDOUT_CACHE) / 'diffusion_planner_training.json'
if not heldout_manifest.exists():
    heldout_npz = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
    heldout_manifest.write_text(json.dumps([f.name for f in heldout_npz]))
print('  [OK] Phase 1b complete')


## 8. GCS sync helper

Mirrors `{DRIVE_OUT_SCALE}/` to `gs://cs269-scale-ablation-archive/scale_ablation_full/`.
Run after every major phase to checkpoint progress.

In [ ]:
GCS_BUCKET = 'cs269-scale-ablation-archive'

def gcs_sync():
    # Try to create the bucket if it doesn't exist
    subprocess.run(['gsutil', 'mb', '-p', 'c269-498020', '-l', 'us-west1', f'gs://{GCS_BUCKET}/'],
                   capture_output=True)
    r = subprocess.run(
        ['gsutil', '-m', 'rsync', '-r',
         f'{DRIVE_OUT_SCALE}/',
         f'gs://{GCS_BUCKET}/scale_ablation_full/'],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        # Only show count of synced files, not all the verbose output
        print(f'  [OK] GCS sync OK')
    else:
        print(f'  [WARN]  GCS sync stderr: {r.stderr[-200:]}')

gcs_sync()
print(f'\n  Bucket: gs://{GCS_BUCKET}/scale_ablation_full/')


## 9. Phase 2 — Train Frenet with best-ever config

Single training: Frenet, seed 269, 5000 scenarios, 50 epochs, batch 32.

**Best-ever Frenet env vars set:**
- `FRENET_SMART_CENTERLINE=1` (Option A route-aware centerline)
- `FRENET_TANH_D=1` + `FRENET_TANH_D_SCALE=3.0` (lateral d clipping)
- Norm stats: `frenet_norm_stats_v1` (paired with tanh)


In [ ]:
NORM_STATS = {
    'waypoints': 'waypoints_norm_stats',
    'frenet':    'frenet_norm_stats_v1',
    'velocity':  'waypoints_norm_stats',
    'acceleration': 'waypoints_norm_stats',
}

def train_and_eval(scale, kin, epochs):
    run_name = f'scale{scale}_{kin}_ep{epochs}_seed{SCALE_SEED}'
    drive_ckpt = f'{DRIVE_OUT_SCALE}/checkpoints/{run_name}.ckpt'
    out_json   = f'{DRIVE_OUT_SCALE}/results/eval_{run_name}.json'

    if pathlib.Path(out_json).exists() and pathlib.Path(drive_ckpt).exists():
        j = json.load(open(out_json))
        print(f'  [OK] already done: ADE={j["ade_mean"]:.2f}')
        return j

    # ===== TRAIN =====
    if not pathlib.Path(drive_ckpt).exists():
        print(f'  TRAINING: {run_name}')
        env = os.environ.copy()
        env['PROJECT_ROOT']         = FP_DIR
        env['SAVE_DIR']             = LOCAL_RUNS
        env['TENSORBOARD_LOG_PATH'] = LOCAL_TB
        env['TRAINING_DATA']        = LOCAL_CACHE
        env['TRAINING_JSON']        = f'{LOCAL_CACHE}/diffusion_planner_training_{scale}.json'
        env['WORLD_SIZE']           = '1'
        env['LOCAL_RANK']           = '0'
        env['MASTER_ADDR']          = 'localhost'
        env['MASTER_PORT']          = '29529'
        env['HYDRA_FULL_ERROR']     = '1'
        env['PYTHONPATH']           = FP_DIR + ':' + env.get('PYTHONPATH', '')

        # Best-ever Frenet config — Option A + tanh clipping
        if kin == 'frenet':
            env['FRENET_SMART_CENTERLINE'] = '1'
            env['FRENET_TANH_D']           = '1'
            env['FRENET_TANH_D_SCALE']     = '3.0'
            print(f'    [frenet] Option A smart centerline + tanh(d/3) clipping ON')
        else:
            for k in ['FRENET_SMART_CENTERLINE', 'FRENET_TANH_D', 'FRENET_TANH_D_SCALE']:
                env.pop(k, None)
        # Clear inference-hook flags (no best-of-N during training)
        for k in ['FRENET_INFERENCE_HOOK', 'FRENET_INFERENCE_K', 'FRENET_INFERENCE_N',
                  'FRENET_INFERENCE_TEMPERATURE']:
            env.pop(k, None)

        if pathlib.Path(LOCAL_RUNS).exists():
            shutil.rmtree(LOCAL_RUNS)
        pathlib.Path(LOCAL_RUNS).mkdir(parents=True, exist_ok=True)

        warmup_e = min(5, max(0, epochs - 1))
        batch = min(TRAIN_BATCH_SIZE, scale)

        cmd = (
            f'{PYTHON} -m torch.distributed.run --nnodes 1 --nproc-per-node 1 --standalone '
            f'{FP_DIR}/flow_planner/trainer.py --config-name flow_planner_standard '
            f'model.kinematic={kin} '
            f'normalization_stats={NORM_STATS[kin]} '
            f'train.batch_size={batch} '
            f'train.epoch={epochs} '
            f'scheduler.warm_up_epoch={warmup_e} '
            f'train.save_utd=1 '
            f'save_every_since={epochs} '
            f'ddp.distributed=false '
            f'seed={SCALE_SEED} '
            f'job_name={run_name} '
            f'num_workers=4'
        )

        log_path = pathlib.Path(LOCAL_RUNS) / f'{run_name}.log'
        with open(log_path, 'w') as logf:
            r = subprocess.run(['bash', '-c', cmd], cwd=FP_DIR, env=env,
                              stdout=logf, stderr=subprocess.STDOUT)

        # Print tail
        with open(log_path) as f:
            tail_lines = f.readlines()[-20:]
        print(''.join(tail_lines))

        if r.returncode != 0:
            print(f'  [FAIL] TRAINING FAILED (exit {r.returncode})')
            raise RuntimeError(f'{run_name} training failed; see {log_path}')

        pths = sorted(pathlib.Path(LOCAL_RUNS).rglob('latest.pth'),
                     key=lambda p: p.stat().st_mtime, reverse=True)
        assert pths, f'no latest.pth produced for {run_name}'
        shutil.copy(pths[0], drive_ckpt)
        shutil.copy(log_path, f'{DRIVE_OUT_SCALE}/logs/{log_path.name}')
        print(f'  [OK] trained -> {drive_ckpt}')

    # ===== EVAL =====
    print(f'  EVAL: {run_name}')
    eval_env = os.environ.copy()
    eval_env['PYTHONPATH'] = FP_DIR + ':' + eval_env.get('PYTHONPATH', '')
    # Propagate Frenet env vars to eval (matches training config)
    if kin == 'frenet':
        eval_env['FRENET_SMART_CENTERLINE'] = '1'
        eval_env['FRENET_TANH_D']           = '1'
        eval_env['FRENET_TANH_D_SCALE']     = '3.0'

    cmd = [
        PYTHON, '-m', 'flow_planner.run_script.inference_eval',
        '--checkpoint',  drive_ckpt,
        '--data_dir',    LOCAL_HELDOUT_CACHE,
        '--data_list',   f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json',
        '--kinematic',   kin,
        '--norm_stats',  NORM_STATS[kin],
        '--output_json', out_json,
        '--batch_size',  str(min(TRAIN_BATCH_SIZE, scale)),
        '--num_batches', '999',
        '--seed',        '0',
        '--no_centerline_encoder',
    ]
    r = subprocess.run(cmd, cwd=FP_DIR, env=eval_env, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [FAIL] EVAL FAILED: {r.stderr[-500:]}')
        raise RuntimeError(f'{run_name} eval failed')

    j = json.load(open(out_json))
    print(f'  [OK] [{run_name}] ADE={j["ade_mean"]:.2f}m  FDE={j["fde_mean"]:.2f}m')
    return j

# Run the (single) combo
results = {}
total = len(SCALE_POINTS) * len(KINEMATICS_TO_RUN) * len(EPOCH_POINTS)
idx = 0
for scale in SCALE_POINTS:
    for kin in KINEMATICS_TO_RUN:
        for epochs in EPOCH_POINTS:
            idx += 1
            print(f'\n[{idx}/{total}] === scale={scale} kin={kin} ep={epochs} ===')
            try:
                results[(scale, kin, epochs)] = train_and_eval(scale, kin, epochs)
            except Exception as e:
                print(f'  [WARN]  combo failed: {e}')
                results[(scale, kin, epochs)] = None
            gcs_sync()

success = sum(1 for v in results.values() if v is not None)
print(f'\n=== Trainings: {success}/{total} success ===')


## 10. Phase 3 — Final report card + GCS sync + download

Writes REPORT_CARD.md summarizing the result. Tarball download for laptop archive.

In [ ]:
# Aggregate results from disk
final_results = {}
for path in sorted(pathlib.Path(f'{DRIVE_OUT_SCALE}/results').glob('eval_scale*.json')):
    name = path.stem.replace('eval_', '')
    parts = name.split('_')
    scale = int(parts[0].replace('scale', ''))
    kin = parts[1]
    epochs = int(parts[2].replace('ep', ''))
    try:
        j = json.load(open(path))
        final_results[(scale, kin, epochs)] = j
    except Exception:
        continue

BASELINE = {
    'waypoints': 4.35,   # multi-seed mean from recover_all
    'frenet':    24.46,  # multi-seed mean from recover_all
}

# Print report
print('=' * 80)
print(f'SCALE ABLATION RESULTS (Frenet best-ever @ {HELDOUT_SCENARIOS_SCALE} heldout)')
print('=' * 80)
for (scale, kin, ep), r in sorted(final_results.items()):
    if r is None: continue
    baseline = BASELINE.get(kin, 0)
    delta = r['ade_mean'] - baseline
    print(f'  {kin:>10} scale={scale:>5} ep={ep:>2}: ADE={r["ade_mean"]:6.2f}m  '
          f'FDE={r["fde_mean"]:6.2f}m  '
          f'(baseline {baseline:.2f}, Δ={delta:+.2f}m)')

# Write Markdown report card
lines = [
    '# Scale Ablation Report Card',
    '',
    f'**Configuration**: Frenet best-ever (Option A smart centerline + tanh(d/3) clipping + frenet_norm_stats_v1)',
    f'**Heldout**: {HELDOUT_SCENARIOS_SCALE} scenarios from val logs (seed-42 log split)',
    f'**Architecture**: Flow Planner (our fork, audit-patched)',
    f'**Training**: batch {TRAIN_BATCH_SIZE}, seed {SCALE_SEED}',
    '',
    '## Results',
    '',
    '| Rep | Scale | Epochs | ADE (m) | FDE (m) | Baseline (1500 scen) | Δ vs baseline |',
    '|---|---|---|---|---|---|---|',
]
for (scale, kin, ep), r in sorted(final_results.items()):
    if r is None: continue
    baseline = BASELINE.get(kin, 0)
    delta = r['ade_mean'] - baseline
    lines.append(f'| {kin} | {scale} | {ep} | {r["ade_mean"]:.2f} | {r["fde_mean"]:.2f} | '
                 f'{baseline:.2f} | {delta:+.2f} |')

# Add conclusion
fr = final_results.get((max(SCALE_POINTS), 'frenet', max(EPOCH_POINTS)))
if fr is not None:
    lines.append('')
    lines.append('## Conclusion')
    lines.append('')
    if fr['ade_mean'] < 15:
        lines.append(f'**Frenet ADE = {fr["ade_mean"]:.2f}m at {max(SCALE_POINTS)} scenarios.** '
                     f'Below 15m threshold -> small-scale penalty was data-limited. '
                     f'Best-ever Frenet config scales up.')
    elif fr['ade_mean'] < 22:
        lines.append(f'**Frenet ADE = {fr["ade_mean"]:.2f}m at {max(SCALE_POINTS)} scenarios.** '
                     f'In the 15-22m range -> partial improvement but representation penalty persists.')
    else:
        lines.append(f'**Frenet ADE = {fr["ade_mean"]:.2f}m at {max(SCALE_POINTS)} scenarios.** '
                     f'Above 20m -> intrinsic-mismatch interpretation confirmed even at scale. '
                     f'The 16.34m best-ever number was an in-train eval artifact.')

pathlib.Path(f'{DRIVE_OUT_SCALE}/REPORT_CARD.md').write_text('\n'.join(lines))
print(f'\n  Wrote REPORT_CARD.md')

# Final GCS sync
gcs_sync()

# Bundle for laptop download
bundle = pathlib.Path(DRIVE_OUT_SCALE)
small_tar = '/content/scale_ablation_small.tar.gz'
small_files = [str(f) for f in bundle.rglob('*') if f.is_file() and f.suffix != '.ckpt']
subprocess.run(['tar', '-czf', small_tar, '--ignore-failed-read'] + small_files, check=False)
print(f'  Small bundle: {small_tar} '
      f'({pathlib.Path(small_tar).stat().st_size/1e6:.1f} MB)')

ckpt_tar = '/content/scale_ablation_ckpts.tar.gz'
ckpts = [str(f) for f in bundle.rglob('*.ckpt')]
if ckpts:
    subprocess.run(['tar', '-czf', ckpt_tar] + ckpts, check=False)
    print(f'  Ckpts bundle: {ckpt_tar} '
          f'({pathlib.Path(ckpt_tar).stat().st_size/1e9:.2f} GB)')

# Try to trigger Colab download — no-op on bare VM
try:
    from google.colab import files
    files.download(small_tar)
    if ckpts:
        time.sleep(3)
        files.download(ckpt_tar)
except (ImportError, ModuleNotFoundError):
    print(f'  [bare-vm] Use gsutil cp to pull tarballs: '
          f'gsutil cp gs://cs269-scale-ablation-archive/scale_ablation_full/ ./')

print(f'\n=== DONE ===')
print(f'GCS bucket: gs://cs269-scale-ablation-archive/scale_ablation_full/')
